# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring a Croissant dataset using the `mlcroissant` library and referencing all dataset elements by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published at: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, their fields, and columns by their `@id` for further exploration.

In [ ]:
# List all record sets with their '@id', name, and fields
print("Available record sets and fields:")
for record_set in dataset.record_sets:
    print(f"• Record Set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '<no name>')}")
    print("  Fields:")
    for field in record_set['fields']:
        print(f"    - Field @id: {field['@id']} ({field.get('name', '<no name>')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

> **Note:** If you are unsure which record set to use, choose the one related to the main data table (usually the largest record set or the one with protein features). For this dataset, we will extract from the main protein data record set.

In [ ]:
# Choose the main record set for protein features.
# Replace this value with the exact '@id' found in the overview step if different.
main_record_set_id = None
for record_set in dataset.record_sets:
    name = record_set.get('name', '').lower()
    if 'protein' in name or 'main' in name or 'vesicle' in name:
        main_record_set_id = record_set['@id']
        break
# Fallback: Use the first available record set
if not main_record_set_id and dataset.record_sets:
    main_record_set_id = dataset.record_sets[0]['@id']

print(f"Using main record set: {main_record_set_id}")

# Optionally, load *all* record sets
all_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set {rs_id} with shape: {df.shape}")

# Inspect columns of the main record set
if main_record_set_id:
    print(f"\nFields/columns for {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record set discovered.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by criteria, normalizing numeric fields, or grouping data by an attribute.

Let's select a quantitative column (e.g., peptide count or molecular weight) referenced by its `@id`.

In [ ]:
# Select a numeric field for analysis
# Update these variable(s) based on the field '@id' and column name from earlier inspection.

# Scan columns to pick out a likely numeric field (e.g., 'Molecular_weight' or 'Peptide_count')
numeric_field = None
sample_df = dataframes[main_record_set_id]
for col in sample_df.columns:
    # Heuristically choose a column likely to be numeric
    if any(key in col.lower() for key in ['molecular', 'weight', 'peptide', 'count', 'mw']):
        numeric_field = col
        break

if not numeric_field:
    numeric_field = sample_df.select_dtypes(include='number').columns[0]

print(f"Selected numeric field: {numeric_field}")

# Filter data where the numeric field is above a threshold
threshold = sample_df[numeric_field].mean() if sample_df[numeric_field].dtype != 'O' else 10
filtered_df = sample_df[sample_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() + 1e-8)
print(f"\nNormalized {numeric_field} for filtered records (first 5):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical feature
group_field = None
for col in filtered_df.columns:
    if col.lower() in ['description', 'gene', 'accession']:
        group_field = col
        break
if not group_field:
    for col in filtered_df.columns:
        if filtered_df[col].dtype == 'O' and col != numeric_field:
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable categorical grouping field found.")

## 5. Visualization
Visualize the distribution and relationships in the filtered data (e.g., normalized numeric field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of Normalized {numeric_field}")
plt.xlabel(f"{numeric_field}_normalized")
plt.show()

# Scatterplot if grouping field is present
if group_field:
    plt.figure(figsize=(12, 5))
    top_cats = filtered_df[group_field].value_counts().nlargest(10).index
    subdf = filtered_df[filtered_df[group_field].isin(top_cats)]
    sns.boxplot(x=group_field, y=numeric_field, data=subdf)
    plt.title(f"{numeric_field} by {group_field} (Top 10 Categories)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset via the Croissant schema with `mlcroissant`. We inspected available record sets using their `@id`, extracted the main protein abundance data, performed basic filtering, normalization, and grouped and visualized main numeric features (such as molecular weight or peptide count). This approach can be adapted to other scientific datasets described with Croissant schemas for reproducible analysis.

For further study, see the [FAIR² dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).